# 0918 15일차

## 1. ImageDataGenerator

폴더에 있는 이미지 파일을 읽어 수치화하고, 변형을 가해 데이터를 늘려주는 도구

**필요한 이유**

1. 이미지를 모델에 넣으려면 `(장수, 세로, 가로, 채널)` 4차원 숫자 배열이어야 함
   - 파일을 하나씩 열고 크기를 맞추고 배열로 바꾸는 일을 대신해줌
2. 이미지 데이터는 양이 부족한 경우가 많음
   - 원본을 뒤집고 돌리고 옮겨서 새로운 데이터처럼 쓰는 것을 **증폭(augmentation)** 이라고 함
   - 원본 파일은 그대로 두고 메모리에서 변형된 이미지를 만들어냄

- 생성할 때 하는 것은 **어떻게 변형할지 설정**뿐이고, 실제 이미지는 아직 읽지 않음

![폴더의 이미지가 flow_from_directory를 거쳐 배치로 묶여 모델로 들어가는 과정](assets/imagedatagenerator-flow.gif)

### 1-1. 증폭 옵션

1. `rescale` : 모든 픽셀에 곱할 값. 0~255를 0~1로 바꾸는 스케일링
2. `horizontal_flip` : 좌우 뒤집기
3. `vertical_flip` : 상하 뒤집기
4. `width_shift_range` / `height_shift_range` : 좌우·상하 평행이동
   - 0.1이면 가로(세로) 길이의 10% 범위 안에서 이동
5. `rotation_range` : 회전 각도. 5면 -5도 ~ +5도
6. `zoom_range` : 확대·축소 비율
7. `shear_range` : 기울이기. 한쪽 좌표를 고정하고 나머지를 밀어 평행사변형처럼 만드는 변형
8. `fill_mode` : 변형으로 생긴 빈 공간을 채우는 방법

- 값은 고정값이 아니라 **범위**이고, 배치를 꺼낼 때마다 그 안에서 랜덤하게 적용됨
- 그래서 같은 이미지라도 epoch마다 조금씩 다른 모습으로 들어감

### 1-2. fill_mode

이미지를 회전·이동·기울이면 원본 픽셀이 없는 빈 공간이 생기는데, 그 자리를 무엇으로 채울지 정하는 옵션

1. `'nearest'` : 가장 가까운 픽셀 값으로 채움 (기본값)
2. `'constant'` : 지정한 상수로 채움 (기본 0 = 검정)
3. `'reflect'` : 경계를 거울처럼 반사해서 채움
4. `'wrap'` : 반대쪽 끝 값을 가져와 채움

- 채우지 않으면 그 자리가 검정으로 남아 원본에 없던 패턴을 학습하게 됨

### 1-3. 훈련 데이터만 증폭하는 이유

증폭 설정은 훈련용과 테스트용을 따로 만들고, 테스트용에는 스케일링만 넣음

1. 증폭은 훈련 데이터를 늘리기 위한 것 → 테스트 데이터를 늘릴 이유가 없음
2. 테스트는 실제로 들어올 데이터를 그대로 평가해야 함 → 변형하면 평가 기준이 흔들림
3. 단, 스케일링은 훈련 때와 똑같이 해줘야 함
   - 훈련은 0~1로 배웠는데 테스트만 0~255면 다른 데이터가 됨

### 1-4. 폴더 구조로 x와 y 만들기

이미지가 담긴 폴더를 지정하면, 그 안의 이미지로 x를 만들고 **하위 폴더 이름으로 y를** 만듦

```
train/
├── ad/      <- 라벨 0
└── normal/  <- 라벨 1
```

1. 라벨은 하위 폴더 이름을 알파벳 순으로 정렬해 0부터 붙임
2. 원본 크기가 제각각이어도 지정한 크기로 맞춰줌
3. 흑백으로 읽으면 채널 1, 컬러로 읽으면 채널 3
4. y의 형태는 이진분류(0/1 한 줄)와 다중분류(원핫) 중 선택

- 따로 라벨을 만들 필요 없이 **폴더를 나눠 두는 것만으로** 분류 데이터가 완성됨
- 몇 장을 몇 개 라벨로 읽었는지 실행할 때 출력해줌

### 1-5. batch_size와 Iterator

읽어온 결과는 numpy 배열이 아니라 **Iterator** — 전체를 메모리에 올리지 않고 요청할 때마다 batch_size만큼 잘라서 내보냄

1. 인덱스는 이미지 번호가 아니라 **배치 번호**
   - 160장을 batch_size 10으로 읽으면 배치는 16개 (0 ~ 15)
   - 없는 배치 번호를 부르면 에러
2. 배치 하나는 `(x, y)` 묶음
   - 앞이 이미지 batch_size장, 뒤가 라벨 batch_size개
   - 이미지 쪽 shape은 `(batch_size, 세로, 가로, 채널)`

**batch_size를 정하는 두 가지 방식**

| 방식 | batch_size | 결과 |
|---|---|---|
| 훈련용 | 10 | 이 값이 곧 훈련 배치 크기가 됨 |
| 통째로 꺼내기 | 전체 장수 이상 | 배치가 1개가 되어 전체 데이터가 한 묶음에 담김 |

- 큰 데이터를 다룰 수 있는 것이 Iterator의 장점
- 반대로 numpy 배열로 저장해두고 쓰려면 통째로 꺼내는 방식(whole batch)을 씀

### 1-6. 제너레이터를 모델에 먹이는 두 가지 방법

1. **통째로 꺼내 쓰기** : 0번 배치를 꺼내 numpy 배열로 만들어 모델에 넘김
   - `batch_size`를 전체 장수 이상으로 줘서 배치를 1개로 만들어야 함
   - 데이터 전체가 메모리에 올라가므로 작은 데이터에서만 가능
   - 일반 numpy 데이터와 똑같이 다룰 수 있어 검증 분리도 그대로 씀
2. **제너레이터를 그대로 넘기기** : 모델이 배치를 하나씩 받아가며 학습
   - 메모리에는 한 배치씩만 올라가므로 전체 크기와 상관없이 돌아감
   - 이때 제너레이터의 `batch_size`가 **곧 훈련 배치 크기**가 됨

- 배치가 여러 개인데 0번만 꺼내면 나머지는 학습에 쓰이지 않음
- 데이터가 작으면 1번이 편하고, 메모리에 다 못 올릴 만큼 크면 2번이 원래 의도된 방식

### 1-7. 주의) 이름이 같은 batch_size 두 개

| 위치 | 의미 |
|---|---|
| 이미지를 읽어올 때 | 제너레이터가 **한 번에 꺼내주는** 이미지 수 |
| 모델을 훈련할 때 | 가중치를 **한 번 갱신하는** 단위 |

1. 통째로 꺼내 쓰면 둘은 별개 — 꺼낸 뒤 다시 작은 단위로 나눠 학습함
2. 제너레이터를 그대로 넘기면 둘은 하나 — 읽어오는 단위가 그대로 갱신 단위가 되므로 훈련 쪽에 따로 지정하지 않음

- 앞의 데이터가 작아서 배치 1개로 끝났다면 이 구분이 드러나지 않음
- 같은 코드를 큰 데이터에 쓰면 일부만 학습되는데, 에러가 나지 않아서 알아차리기 어려움

### 1-8. 검증 데이터를 나누는 방법

제너레이터에는 훈련할 때 쓰던 **검증 비율 지정이 통하지 않음** — 배치 단위로 들어오기 때문에 전체를 비율로 자를 수 없음

1. 증폭 설정을 만들 때 검증 비율을 함께 지정
2. 같은 폴더를 두 번 읽되 한쪽은 훈련용, 한쪽은 검증용으로 구분해서 받음

- 비율과 폴더가 같아야 두 쪽이 겹치지 않게 갈라짐
- 검증용을 따로 두지 않고 테스트 데이터를 검증에 쓰면, 테스트가 더 이상 처음 보는 데이터가 아니게 됨

### 1-9. 주의) 평가용 데이터는 섞지 않는다

예측 결과는 제너레이터가 내보낸 **순서대로** 나옴

1. 섞어서 읽으면 예측값의 순서와 원래 정답의 순서가 달라짐
2. 정확도를 따로 계산할 때 엉뚱한 정답과 비교하게 됨

- 배치로 꺼내 쓸 때는 x와 y가 같이 나오므로 섞여도 짝이 맞음
- 예측만 따로 뽑아 정답과 비교할 때 순서가 문제가 됨